# Real-model runbook — GCG / PAIR attacks on free-tier GPU

**Where this runs:** Kaggle notebook, accelerator `GPU T4 x2`, Internet `On`.

**Cost:** £0. Kaggle gives 30 GPU-hours per week; the whole experiment matrix fits in
one week's quota at 1-2B model sizes.

**Order matters.** Cell 3 measures seconds/step *before* you spend hours on a
real run. Skipping it is how a session dies at hour 11 with nothing saved.

**What "done" looks like:** at least one cell prints a non-zero ASR. Every ASR
you have seen so far in this project was 0%, because the model under attack
was a 2-layer *random* model with no refusal behaviour to break. This is the
first time the attack is aimed at something that can actually refuse.

Read `kaggle/README.md` for the account/secret setup this notebook assumes.


## 1. Setup

In [ ]:
# Only install what Kaggle does not already ship. Installing torch would
# waste ~10 minutes of your session for no reason.
!pip install -q bitsandbytes accelerate datasets 2>&1 | tail -3

import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("device count  :", torch.cuda.device_count())


In [ ]:
!nvidia-smi


### Clone the repo

Change `REPO_URL` to your repository. If you set up a Kaggle secret called
`GITHUB_TOKEN` (see `kaggle/README.md`), the last line authenticates the
remote so you can push results straight back to GitHub.

In [ ]:
REPO_URL = "https://github.com/Rohit4507/YOUR-REPO.git"   # <-- EDIT ME

import os, subprocess

# Kaggle starts in /kaggle/working, which is the ONLY directory that survives
# "Save Version". Clone into it, not /tmp.
%cd /kaggle/working
if not os.path.isdir("/kaggle/working/repo"):
    !git clone $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3


In [ ]:
# Optional: wire up git push-back so runs commit themselves.
git_name  = "Rohit Singh Rajput"
git_email = "rohitrajput33220@gmail.com"

!git config user.name  "{git_name}"
!git config user.email "{git_email}"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.system(f"git remote set-url origin https://{token}@github.com/Rohit4507/YOUR-REPO.git")
    PUSH_ENABLED = True
    print("git push-back ENABLED (GITHUB_TOKEN found)")
except Exception as e:
    PUSH_ENABLED = False
    print("git push-back disabled:", type(e).__name__)
    print("Results still land in /kaggle/working — use 'Save Version' to keep them.")


## 2. Offline checks

Seconds on CPU. These prove the *code* is correct. They say nothing about
whether the attack works — that is the next section.

In [ ]:
!python -m experiments.run_smoke 2>&1 | tail -14
print()
!python -m unittest discover -s tests 2>&1 | tail -4


## 3. Measure seconds/step FIRST

`--steps 5` on a real model. Multiply by 500 to get the real cost. If one step
takes 60s, 500 steps is 8 hours *per prompt* and the 25-prompt table in the
plan will not fit in a week — you would need to drop to a smaller model or
fewer prompts. Decide that now, not at hour 10.

Expect the first call to pause ~2 minutes while bitsandbytes sets up 4-bit.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

import time
t0 = time.time()
!python -m experiments.run_algorithm1 --preset paper --benchmark \
    --model {MODEL} --load-in-4bit --steps 5 --num-prompts 2
print(f"\nWALL CLOCK for 5 steps: {time.time() - t0:.0f}s")
print("Extrapolate: seconds/step x 500 = cost of a real run.")


## 4. GATE 1 — first real attack

This is the whole point of the exercise. Watch two numbers:

* **loss** — should fall well below the `random-guess floor` line printed at the
  start. On a real model that floor is ~11.9 (`ln(151936)`), and a working
  suffix pushes the loss toward 1-2. If the loss sits at the floor, the attack
  is not biting.
* **ASR** — the first non-zero number in this project's history.

In [ ]:
!python -m experiments.run_algorithm1 --preset paper --benchmark \
    --model {MODEL} --load-in-4bit --steps 500 --num-prompts 10 \
    2>&1 | tee /kaggle/working/logs_algorithm1.txt | tail -30


## 5. Universal attack (Algorithm 2) — make this your primary result

One suffix, many prompts, one optimisation. Far cheaper per prompt than
attacking each prompt separately, which is what makes a real table affordable
on free tier.

Held-out prompts are scored with the suffix they never saw — that is the
generalisation number worth publishing.

In [ ]:
!python -m experiments.run_algorithm2 --preset paper --benchmark \
    --model {MODEL} --load-in-4bit --steps 500 \
    --num-prompts 25 --held-out 5 \
    2>&1 | tee /kaggle/working/logs_algorithm2.txt | tail -20


## 6. Cross-model transfer

Optimise on the source, carry the suffix over as *text*, re-tokenise for the
target. `--load-in-4bit` is applied to both models (it used to be missing here,
which silently blocked 7-8B models on a 16GB card).

On a T4 x2 session the two models fit side by side, so this is one cell, not
two.

In [ ]:
SOURCE = "Qwen/Qwen2.5-1.5B-Instruct"
TARGET = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

!python -m experiments.run_transfer --preset paper --benchmark \
    --model {SOURCE} --target-model {TARGET} --load-in-4bit \
    --steps 500 --num-prompts 10 \
    2>&1 | tee /kaggle/working/logs_transfer.txt | tail -18


## 7. Multi-turn (3-turn) attack

The plan asks for a 3-turn variant; nothing in the repo did this before.

Suffix is optimised against the goal as the **third user turn** of a scripted
escalation conversation, using the model's own replies as context. Reports
single-turn vs multi-turn ASR side by side.

**A null result is a real result** if you report it as one. Do not dress up
"both zero" as a finding.

In [ ]:
!python -m experiments.run_multiturn --preset paper --benchmark \
    --model {MODEL} --load-in-4bit --steps 500 \
    --num-prompts 10 --n-prior-turns 2 \
    2>&1 | tee /kaggle/working/logs_multiturn.txt | tail -20


## 8. Baselines — never publish an ASR without one

Random search, identical budget. If GCG cannot beat random search, the
"gradient-guided" claim in every summary of this work is unsupported.

In [ ]:
!python -m experiments.run_baselines --preset paper --benchmark \
    --model {MODEL} --load-in-4bit --steps 500 --num-prompts 10 \
    2>&1 | tee /kaggle/working/logs_baselines.txt | tail -16


## 9. Stop and collect

Before the session dies: save outputs, tally the ASR table, push to git.

**Budget checkpoint.** If you have used more than ~20 of your 30 weekly hours,
stop here and Continue the session next week rather than burning the remaining
quota on a half-run experiment.

In [ ]:
from collections import Counter
import glob, json

rows = []
for path in sorted(glob.glob("results/reports/generated/*.json")):
    d = json.load(open(path))
    kind = d.get("run_kind", "legacy")
    label = (d.get("model") or {}).get("model_label")             or (d.get("source_model") or {}).get("model_label")
    acc = d.get("accounting", {})
    summary = (d.get("evaluation", {}) or {}).get("summary")               or (d.get("in_domain", {}) or {}).get("summary") or {}
    rows.append({
        "file": path.split("/")[-1],
        "kind": kind,
        "model": label,
        "asr": summary.get("asr_prefix_match"),
        "forwards": acc.get("total_forwards"),
        "successes": acc.get("n_successes"),
        "cost_per_success": acc.get("cost_per_success_usd"),
    })

for r in rows:
    print(f"{r['kind'][:26]:28s} | {str(r['asr']):>6s} | "
          f"{str(r['forwards']):>10s} | {r['file']}")

nonzero = [r for r in rows if (r["asr"] or 0) > 0]
print()
print(f"runs with non-zero ASR: {len(nonzero)} / {len(rows)}")
if not nonzero:
    print("Still zero everywhere. Check the loss against the random-guess "
          "floor before concluding anything about the attack.")


In [ ]:
!cp -r results /kaggle/working/ 2>/dev/null
for f in glob.glob("/kaggle/working/logs_*.txt"):
    print(f)
print()
print("Files under /kaggle/working survive 'Save Version'. Anything outside it does not.")


In [ ]:
if PUSH_ENABLED:
    !git add -A
    !git commit -m "Real-model runs from Kaggle: algorithm1/2, transfer, multiturn, baselines" || echo "nothing to commit"
    !git push
    print("Pushed to GitHub.")
else:
    print("Push disabled. Use: File > Save Version (Quick Save), then download")
    print("the Output files, or add a GITHUB_TOKEN secret and re-run cell 4.")


## 10. What to do with the numbers

1. **Results table with n and a CI.** Every cell needs the number of prompts it
   is over. A rate without `n` is not a result.
2. **Cost.** `cost_per_success_usd` is `null` when nothing succeeded — that is
   deliberate; it means "undefined", not "free". On free tier report
   `total_forwards` and `forwards_per_success`, which are hardware-independent.
3. **Report what you did not run.** "We report 1-2B models; 7-8B was out of
   compute budget for an unfunded independent project" is a strength.
   Silently shrinking the claim is not.
4. Then write it up and link this notebook's output from the post.
